# Mosquito Season Change: 1961–1990 vs 1991–2020

**Project:** When is mosquito season in your city?  
**Author:** Andrés Lill  
**Purpose:** Quantify how modelled climate suitability for mosquito activity has shifted between the two WMO climatological reference periods.  
**Output:** `mosquito_suitability_delta.csv`: one row per city with modelled season lengths for both periods and the delta.

**Requires:** `mosquito_suitability.csv` (1991–2020 baseline, already computed) in the same directory.

> **Important:** Changes reflect modelled climate suitability under two climatological reference periods, not observed mosquito range shifts or confirmed changes in distribution.

## 1. Setup

> **Note:** This notebook is designed to run on Google Colab and reads the CDS API key from Colab Secrets (`CDSAPI_KEY`). To run locally, replace the `userdata.get()` call with `os.environ.get('CDSAPI_KEY')` or write your key directly to `~/.cdsapirc`.

In [2]:
import os
import pathlib

try:
    from google.colab import userdata
    key = userdata.get("CDSAPI_KEY")
except ImportError:
    key = os.environ.get("CDSAPI_KEY")

assert key, "CDSAPI_KEY not found. Set it in Colab Secrets or as an environment variable."
pathlib.Path("/root/.cdsapirc").write_text(
    "url: https://cds.climate.copernicus.eu/api\n"
    f"key: {key}\n"
)

85

In [3]:
!pip install cdsapi xarray netcdf4 tqdm scipy --quiet

import cdsapi
c = cdsapi.Client()
print("Connected")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.9 MB/s eta 0:00:00
Connected


## 2. Download ERA5 1961–1990

Same variables as the 1991–2020 download. The CDS API supports ERA5-back-extension for years 1950–1978 and standard ERA5 from 1979 onwards, both are covered transparently by the same dataset name.

**Note:** This job covers 30 years × 12 months = 360 time steps. Expect ~5–15 min queue + download time on CDS.

In [4]:
OUTPUT_DIR      = "./"  # e.g. "/content/drive/MyDrive/Mosquito Project/Data/"
OUTPUT_PATH_OLD = os.path.join(OUTPUT_DIR, "era5_monthly_1961_1990.nc")

os.makedirs(OUTPUT_DIR, exist_ok=True)

c.retrieve(
    "reanalysis-era5-single-levels-monthly-means",
    {
        "product_type": "monthly_averaged_reanalysis",
        "variable": [
            "2m_temperature",
            "2m_dewpoint_temperature",
            "total_precipitation",
        ],
        "year": [str(y) for y in range(1961, 1991)],
        "month": ["01","02","03","04","05","06","07","08","09","10","11","12"],
        "time": "00:00",
        "format": "netcdf",
    },
    OUTPUT_PATH_OLD,
)
print(f"Download complete → {OUTPUT_PATH_OLD}")

2026-04-24 21:30:13,802 INFO Request ID is 81da6583-9ce3-4722-9ac3-a0937f8e6087
INFO:ecmwf.datastores.legacy_client:Request ID is 81da6583-9ce3-4722-9ac3-a0937f8e6087
2026-04-24 21:30:13,953 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-04-24 21:30:23,987 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-04-24 21:34:35,522 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


318b8ef3b4fe5233928e9b0cb3f91dad.zip:   0%|          | 0.00/1.37G [00:00<?, ?B/s]

Download complete → ./era5_monthly_1961_1990.nc


## 3. Model Functions

Identical to the 1991–2020 pipeline. No model changes. Only the ERA5 climatology input differs.

In [5]:
import math
import numpy as np
import pandas as pd
import xarray as xr
from scipy import stats
from tqdm import tqdm

TEMP_PARAMS = {
    "aegypti":    (14.97, 27.1,  39.15),
    "albopictus": (11.02, 24.5,  38.07),
}
VPD_LOW, VPD_HIGH = 1.0, 3.0
SEASON_THRESHOLD  = 0.3

# Photoperiod parameters — updated to match mosquito_suitability_pipeline.ipynb
CPP_LAT_MIN      = 25.0
CPP_LAT_MAX      = 40.0
CPP_MIN          = 12.3   # Xia et al. 2018: Guangzhou subtropical population
CPP_MAX          = 13.5   # Lacour et al. 2015: French Mediterranean population
CPP_STEEPNESS    = 8.0
PHOTO_INFLECTION = 23.5
PHOTO_K_LAT      = 0.5


def temp_score_triangle(temp_c, tmin, topt, tmax):
    if temp_c <= tmin or temp_c >= tmax: return 0.0
    if temp_c <= topt: return (temp_c - tmin) / (topt - tmin)
    return (tmax - temp_c) / (tmax - topt)

def calc_vpd_kpa(temp_c, rh_pct):
    svp = 0.6108 * math.exp((17.27 * temp_c) / (temp_c + 237.3))
    return max(0.0, svp * (1.0 - rh_pct / 100.0))

def vpd_score(vpd_kpa):
    if vpd_kpa <= VPD_LOW: return 1.0
    if vpd_kpa >= VPD_HIGH: return 0.0
    return (VPD_HIGH - vpd_kpa) / (VPD_HIGH - VPD_LOW)

def photoperiod_hours(lat_deg, month):
    doy_mid = [15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349]
    doy = doy_mid[month - 1]
    lat = math.radians(lat_deg)
    decl = math.radians(23.44) * math.sin(math.radians(360 / 365.0 * (doy - 81)))
    cos_ha = max(-1.0, min(1.0, -math.tan(lat) * math.tan(decl)))
    return round(2 * math.degrees(math.acos(cos_ha)) / 15.0, 2)

def albopictus_cpp(abs_lat):
    """
    Latitude-dependent critical photoperiod (CPP).
    Linear interpolation from CPP_MIN at CPP_LAT_MIN to CPP_MAX at CPP_LAT_MAX.
    """
    lat_eff = np.clip(abs_lat, CPP_LAT_MIN, CPP_LAT_MAX)
    return CPP_MIN + (lat_eff - CPP_LAT_MIN) * (CPP_MAX - CPP_MIN) / (CPP_LAT_MAX - CPP_LAT_MIN)

def albopictus_photo_factor(daylen_h, lat_deg):
    """
    Two-stage continuous photoperiod model (updated).
    Stage 1: sigmoid latitude weight (inflection=23.5°, k=0.5)
    Stage 2: logistic response around latitude-dependent CPP (steepness=8.0)
    Matches mosquito_suitability_pipeline.ipynb.
    """
    abs_lat = abs(lat_deg)
    latitude_weight = 1.0 / (1.0 + np.exp(-PHOTO_K_LAT * (abs_lat - PHOTO_INFLECTION)))
    cpp = albopictus_cpp(abs_lat)
    photoperiod_response = 1.0 / (1.0 + np.exp(-CPP_STEEPNESS * (daylen_h - cpp)))
    return float(np.clip(1.0 - latitude_weight * (1.0 - photoperiod_response), 0.0, 1.0))

def days_in_month(month):
    return [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31][month - 1]

print("Model functions loaded.")


Model functions loaded.


## 4. Compute Suitability for 1961–1990

In [6]:
import glob
print(glob.glob("/content/**/*.nc", recursive=True))

['/content/era5_monthly_1961_1990.nc']


In [7]:
import zipfile

with zipfile.ZipFile("/content/era5_monthly_1961_1990.nc", 'r') as z:
    z.extractall("/content/era5_1961_1990/")
    print(z.namelist())

['data_stream-moda_stepType-avgua.nc', 'data_stream-moda_stepType-avgad.nc']


In [8]:
ds_old1 = xr.open_dataset("/content/era5_1961_1990/data_stream-moda_stepType-avgua.nc")
ds_old2 = xr.open_dataset("/content/era5_1961_1990/data_stream-moda_stepType-avgad.nc")

t2m_clim = ds_old1["t2m"].groupby("valid_time.month").mean("valid_time")
d2m_clim = ds_old1["d2m"].groupby("valid_time.month").mean("valid_time")
tp_clim  = ds_old2["tp"].groupby("valid_time.month").mean("valid_time")

print(f"Climatology shape: {t2m_clim.shape}  (expected: 12 × lat × lon)")


Climatology shape: (12, 721, 1440)  (expected: 12 × lat × lon)


In [9]:
EXTRACT_DIR = "/content/era5_1961_1990/"

ds_old1 = xr.open_dataset(os.path.join(EXTRACT_DIR, "data_stream-moda_stepType-avgua.nc"))
ds_old2 = xr.open_dataset(os.path.join(EXTRACT_DIR, "data_stream-moda_stepType-avgad.nc"))

t2m_clim = ds_old1["t2m"].groupby("valid_time.month").mean("valid_time")
d2m_clim = ds_old1["d2m"].groupby("valid_time.month").mean("valid_time")
tp_clim  = ds_old2["tp"].groupby("valid_time.month").mean("valid_time")

print(f"Climatology shape: {t2m_clim.shape}  (expected: 12 × lat × lon)")

Climatology shape: (12, 721, 1440)  (expected: 12 × lat × lon)


In [10]:
# df_baseline loads the existing 1991–2020 CSV (mosquito_suitability.csv).
# It is used here ONLY to extract the city list (city, country, lat, lon).
# Suitability scores are NOT carried over, they are recomputed
# from the ERA5 1961–1990 climatology (t2m_clim, d2m_clim, tp_clim).
df_baseline = pd.read_csv("mosquito_suitability.csv")
cities = (
    df_baseline[["city", "country", "lat", "lon"]]
    .drop_duplicates(subset=["city", "country"])
    .reset_index(drop=True)
)
print(f"{len(cities)} cities loaded from baseline CSV")

rows = []

for _, city in tqdm(cities.iterrows(), total=len(cities), desc="Cities"):
    lat, lon = float(city["lat"]), float(city["lon"])
    lon360 = lon if lon >= 0 else lon + 360

    t2m = t2m_clim.sel(latitude=lat, longitude=lon360, method="nearest").values
    d2m = d2m_clim.sel(latitude=lat, longitude=lon360, method="nearest").values

    temp_c = t2m - 273.15
    dewp_c = d2m - 273.15
    rh = 100 * np.exp((17.27 * dewp_c) / (dewp_c + 237.3)) / \
               np.exp((17.27 * temp_c) / (temp_c + 237.3))
    rh = np.clip(rh, 0, 100)

    for i, month in enumerate(range(1, 13)):
        t   = float(temp_c[i])
        rh_ = float(rh[i])
        vpd    = calc_vpd_kpa(t, rh_)
        vs     = vpd_score(vpd)
        daylen = photoperiod_hours(lat, month)
        ts_aeg = temp_score_triangle(t, *TEMP_PARAMS["aegypti"])
        ts_alb = temp_score_triangle(t, *TEMP_PARAMS["albopictus"])
        pf_alb = albopictus_photo_factor(daylen, lat)

        rows.append({
            "city":    city["city"],
            "country": city["country"],
            "lat":     round(lat, 4),
            "lon":     round(lon, 4),
            "month":   month,
            "suitability_score_aegypti":    round(ts_aeg * vs, 4),
            "suitability_score_albopictus": round(ts_alb * vs * pf_alb, 4),
        })

df_old = pd.DataFrame(rows)
print(f"Done. {len(df_old)} rows computed ({len(df_old) // 12} cities × 12 months).")

1423 cities loaded from baseline CSV


Cities: 100%|██████████| 1423/1423 [00:03<00:00, 416.22it/s]

Done. 17076 rows computed (1423 cities × 12 months).


## 5. Compute Delta

Modelled suitable season length = number of months where suitability score ≥ 0.3.  
Delta = season_1991_2020 − season_1961_1990  
Positive delta = modelled suitable season length increased. Negative = decreased.

In [11]:
season_old = (
    df_old.groupby(["city", "country", "lat", "lon"])
    .apply(lambda x: pd.Series({
        "season_aegypti_1961_1990":    int((x["suitability_score_aegypti"]    >= SEASON_THRESHOLD).sum()),
        "season_albopictus_1961_1990": int((x["suitability_score_albopictus"] >= SEASON_THRESHOLD).sum()),
    }))
    .reset_index()
)

season_new = (
    df_baseline.groupby(["city", "country", "lat", "lon"])
    .apply(lambda x: pd.Series({
        "season_aegypti_1991_2020":    int((x["suitability_score_aegypti"]    >= SEASON_THRESHOLD).sum()),
        "season_albopictus_1991_2020": int((x["suitability_score_albopictus"] >= SEASON_THRESHOLD).sum()),
    }))
    .reset_index()
)

df_delta = season_new.merge(season_old, on=["city", "country", "lat", "lon"], how="left")

df_delta["delta_aegypti"]    = df_delta["season_aegypti_1991_2020"]    - df_delta["season_aegypti_1961_1990"]
df_delta["delta_albopictus"] = df_delta["season_albopictus_1991_2020"] - df_delta["season_albopictus_1961_1990"]
df_delta["abs_lat"]          = df_delta["lat"].abs()

# Mean annual temperature from baseline (for correlation analysis and Tableau)
mean_temp_lookup = df_baseline.groupby(["city", "country"])["temp_mean_c"].mean().reset_index()
df_delta = df_delta.merge(mean_temp_lookup, on=["city", "country"], how="left")

print(df_delta[[
    "city", "country",
    "season_aegypti_1961_1990", "season_aegypti_1991_2020", "delta_aegypti",
    "season_albopictus_1961_1990", "season_albopictus_1991_2020", "delta_albopictus"
]].head(10).to_string(index=False))
print(f"\nRows: {len(df_delta)}")

/tmp/ipykernel_1324/1909129057.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


              city              country  season_aegypti_1961_1990  season_aegypti_1991_2020  delta_aegypti  season_albopictus_1961_1990  season_albopictus_1991_2020  delta_albopictus
               Aba              Nigeria                        12                        12              0                           12                           12                 0
          Abeokuta              Nigeria                        12                        12              0                           12                           12                 0
           Abhepur                India                         5                         5              0                            4                            4                 0
           Abidjan        Côte d’Ivoire                        12                        12              0                           12                           12                 0
     Abomey-Calavi                Benin                        12                    

/tmp/ipykernel_1324/1909129057.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


## 6. Summary Statistics

In [12]:
for sp, col, baseline_col in [
    ("aegypti",    "delta_aegypti",    "season_aegypti_1991_2020"),
    ("albopictus", "delta_albopictus", "season_albopictus_1991_2020"),
]:
    print(f"=== {sp} ===")
    print(df_delta[col].describe().round(2))
    print(f"  Modelled season increased:  {(df_delta[col] > 0).sum()}")
    print(f"  Unchanged:                  {(df_delta[col] == 0).sum()}")
    print(f"  Modelled season decreased:  {(df_delta[col] < 0).sum()}")
    print()
    print(f"  Spearman correlations with {col}:")
    for label, xcol in [
        ("abs_lat",        "abs_lat"),
        ("mean_temp",      "temp_mean_c"),
        ("baseline season", baseline_col),
    ]:
        r, p = stats.spearmanr(df_delta[xcol], df_delta[col])
        print(f"    {label:20s}: r={r:6.3f}, p={p:.4f}")
    print()

=== aegypti ===
count    1423.00
mean        0.30
std         0.65
min        -3.00
25%         0.00
50%         0.00
75%         1.00
max         5.00
Name: delta_aegypti, dtype: float64
  Modelled season increased:  387
  Unchanged:                  1000
  Modelled season decreased:  36

  Spearman correlations with delta_aegypti:
    abs_lat             : r= 0.132, p=0.0000
    mean_temp           : r=-0.163, p=0.0000
    baseline season     : r=-0.081, p=0.0022

=== albopictus ===
count    1423.00
mean        0.08
std         0.47
min        -3.00
25%         0.00
50%         0.00
75%         0.00
max         3.00
Name: delta_albopictus, dtype: float64
  Modelled season increased:  158
  Unchanged:                  1210
  Modelled season decreased:  55

  Spearman correlations with delta_albopictus:
    abs_lat             : r= 0.082, p=0.0021
    mean_temp           : r=-0.180, p=0.0000
    baseline season     : r= 0.077, p=0.0037



## 7. Export

In [13]:
out_path = "mosquito_suitability_delta.csv"
df_delta.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Columns: {df_delta.columns.tolist()}")
print(f"Rows: {len(df_delta)}")

Saved: mosquito_suitability_delta.csv
Columns: ['city', 'country', 'lat', 'lon', 'season_aegypti_1991_2020', 'season_albopictus_1991_2020', 'season_aegypti_1961_1990', 'season_albopictus_1961_1990', 'delta_aegypti', 'delta_albopictus', 'abs_lat', 'temp_mean_c']
Rows: 1423
